# Avance 2: Análisis de Trayectoria en la Fórmula 1 (1950–2023)
**Integrantes:** Andy Villarroel, Javier Alcaino  
**Curso:** Análisis de Datos e Inferencia Estadística  
**Fecha:** Mayo 2025

## 1. Introducción y Pregunta de Investigación

En la Fórmula 1, el talento del piloto interactúa directamente con el rendimiento del monoplaza. Históricamente, los equipos con mayores recursos financieros y tecnológicos —como Ferrari, McLaren o Mercedes— han dominado el campeonato de forma recurrente. Esto plantea una pregunta estructural sobre la equidad del deporte:

> **¿El promedio de puntos por carrera (PPC) de los pilotos que debutaron en equipos Top es significativamente mayor que el de aquellos que debutaron en equipos chicos?**

**Variable dependiente:** Puntos Por Carrera (PPC) = puntos totales / número de carreras disputadas.  
**Variable independiente principal:** Tipo de equipo de debut (Top = ganador del Campeonato de Constructores WCC, Chico = resto).  
**Variable de control:** Número total de carreras disputadas (proxy de experiencia).

**Hipótesis general:** Los pilotos que debutan en equipos Top tienen un PPC significativamente mayor, debido a que acceden a mejores monoplazas desde el inicio de su carrera.

## 2. Carga de Librerías y Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

sns.set_theme(style='whitegrid', font_scale=1.1)
color_dict = {'Top': '#E10600', 'Chico': '#3A86FF'}

## 3. Descripción del Dataset

Los datos provienen de la base de datos pública **Ergast Motor Racing Database** (disponible en Kaggle: *Formula 1 World Championship 1950–2023*). Contiene registros históricos completos de la Fórmula 1 desde la primera temporada.

| Tabla | Descripción | Unidad de análisis |
|---|---|---|
| `results.csv` | Resultado por piloto por carrera (puntos, posición, vueltas) | Resultado individual |
| `races.csv` | Información de cada carrera (año, circuito, fecha) | Carrera |
| `drivers.csv` | Datos personales de pilotos | Piloto |
| `constructors.csv` | Información de escuderías | Constructor |
| `constructor_standings.csv` | Clasificación de constructores al final de cada carrera | Constructor × Carrera |

**Tipo de datos:** Longitudinales (panel de datos deportivos 1950–2023).  
**Variables principales:**
- `ppc`: Puntos Por Carrera — variable dependiente cuantitativa continua
- `tipo_debut`: Top o Chico — variable independiente categórica binaria
- `n_carreras`: Total de carreras disputadas — variable de control cuantitativa
- `equipo_debut`: Nombre del equipo donde debutó el piloto

In [ ]:
races = pd.read_csv('races.csv', na_values='\\N')
drivers = pd.read_csv('drivers.csv', na_values='\\N')
constructors = pd.read_csv('constructors.csv', na_values='\\N')
results = pd.read_csv('results.csv', na_values='\\N')
constructor_std = pd.read_csv('constructor_standings.csv', na_values='\\N')

print('Dimensiones de cada tabla:')
for nombre, tabla in [('races', races), ('drivers', drivers), ('constructors', constructors),
                       ('results', results), ('constructor_standings', constructor_std)]:
    print(f'  {nombre}: {tabla.shape[0]:,} filas x {tabla.shape[1]} columnas')

## 4. Limpieza y Preparación de Datos

En esta sección documentamos cada decisión metodológica tomada durante el preprocesamiento.

In [ ]:
# --- 4.1 Revisión de valores nulos ---
print('=== Valores nulos en results (columnas relevantes) ===')
cols_relevantes = ['driverId', 'constructorId', 'points', 'raceId']
print(results[cols_relevantes].isnull().sum())

# --- 4.2 Revisión de duplicados ---
print(f'\nFilas duplicadas en results: {results.duplicated().sum()}')
print(f'Filas duplicadas en drivers: {drivers.duplicated().sum()}')

# --- 4.3 Tipos de datos ---
print('\n=== Tipos de datos en results (columnas clave) ===')
print(results[cols_relevantes].dtypes)

# --- 4.4 Distribución de puntos ---
print(f'\nRegistros con points = 0: {(results["points"] == 0).sum():,} ({(results["points"] == 0).mean()*100:.1f}%)')
print(f'Rango de puntos: {results["points"].min()} - {results["points"].max()}')

In [ ]:
# --- 4.5 Definición de equipos Top (ganadores del WCC) ---
# Se considera 'Top' a todo equipo que haya ganado al menos 1 Campeonato de Constructores.
# Se toma la última carrera de cada año como representativa del campeón final.
ultima_carrera_x_anio = races.sort_values('date').groupby('year')['raceId'].last().reset_index(name='raceId')
ids_top = constructor_std.merge(ultima_carrera_x_anio, on='raceId').query('position == 1')['constructorId'].unique()
constructors['tipo'] = np.where(constructors['constructorId'].isin(ids_top), 'Top', 'Chico')

print('Equipos Top (con al menos 1 WCC):')
print(constructors[constructors['tipo'] == 'Top']['name'].values)

# --- 4.6 Equipo de debut de cada piloto ---
# Se define debut como la primera aparición del piloto en results, ordenada por raceId (proxy de orden cronológico).
debut = results.sort_values('raceId').groupby('driverId').first().reset_index()[['driverId', 'constructorId']]
debut = debut.rename(columns={'constructorId': 'constructor_debut'})
debut = debut.merge(constructors[['constructorId', 'name', 'tipo']],
                    left_on='constructor_debut', right_on='constructorId')
debut = debut.rename(columns={'name': 'equipo_debut', 'tipo': 'tipo_debut'})

# --- 4.7 Cálculo del PPC (mínimo 10 carreras) ---
# Se excluyen pilotos con < 10 carreras para evitar que participaciones esporádicas
# distorsionen el promedio (ej. un piloto que corrió 1 carrera con podio tendría PPC altísimo).
puntos_por_carrera = results.groupby('driverId').agg(
    pts_totales=('points', 'sum'),
    n_carreras=('raceId', 'count')
).reset_index()
n_antes = puntos_por_carrera.shape[0]
puntos_por_carrera = puntos_por_carrera.query('n_carreras >= 10')
n_despues = puntos_por_carrera.shape[0]
print(f'\nPilotos excluidos por < 10 carreras: {n_antes - n_despues} ({((n_antes-n_despues)/n_antes)*100:.1f}% del total)')

puntos_por_carrera['ppc'] = puntos_por_carrera['pts_totales'] / puntos_por_carrera['n_carreras']

# --- 4.8 Dataset final ---
df = puntos_por_carrera.merge(debut[['driverId', 'equipo_debut', 'tipo_debut']], on='driverId')
df = df.merge(drivers[['driverId', 'forename', 'surname']], on='driverId')
df['nombre'] = df['forename'] + ' ' + df['surname']

print(f'\nDataset final: {len(df)} pilotos analizados')
print(f'  Debut en Top:   {(df["tipo_debut"] == "Top").sum()} pilotos ({(df["tipo_debut"] == "Top").mean()*100:.1f}%)')
print(f'  Debut en Chico: {(df["tipo_debut"] == "Chico").sum()} pilotos ({(df["tipo_debut"] == "Chico").mean()*100:.1f}%)')

**Decisiones metodológicas clave:**

1. **Sin duplicados ni nulos** en las columnas clave (`driverId`, `constructorId`, `points`): no fue necesario eliminar registros por este criterio.
2. **Definición de 'Top'**: se utilizó el criterio objetivo de haber ganado al menos un Campeonato de Constructores (WCC). Esto excluye equipos competitivos pero sin título (ej. Brabham en sus inicios), lo que podría subestimar el grupo Top.
3. **Filtro de 10 carreras mínimas**: elimina aproximadamente un tercio de los pilotos registrados. Esta decisión es conservadora y válida, ya que impide que resultados atípicos de participaciones cortas distorsionen el PPC promedio.
4. **Ordenamiento por raceId para el debut**: se asume que los raceId son incrementales y representan orden cronológico, lo cual es válido para esta base de datos.
5. **El sistema de puntuación no está estandarizado**: esta es la limitación más importante y se discute en la sección 7.

## 5. Análisis Exploratorio de Datos (EDA)

### 5.1 Estadística Descriptiva

In [ ]:
print('=== Estadística Descriptiva del PPC por Tipo de Debut ===')
desc = df.groupby('tipo_debut')['ppc'].describe().round(4)
display(desc)

print('\n=== Top 5 pilotos con mayor PPC (Debut Top) ===')
display(df[df['tipo_debut'] == 'Top'].nlargest(5, 'ppc')[['nombre', 'equipo_debut', 'n_carreras', 'ppc']])

print('\n=== Top 5 pilotos con mayor PPC (Debut Chico) ===')
display(df[df['tipo_debut'] == 'Chico'].nlargest(5, 'ppc')[['nombre', 'equipo_debut', 'n_carreras', 'ppc']])

**Interpretación de la estadística descriptiva:**

La tabla revela diferencias sustanciales entre ambos grupos. El grupo **Top** presenta una media de PPC considerablemente mayor que el grupo **Chico**. Más revelador aún es que la **mediana del grupo Top supera la media del grupo Chico**, indicando que incluso el piloto 'típico' de debut élite supera ampliamente al promedio del grupo rival.

El grupo Top también exhibe mayor dispersión (desviación estándar más alta), lo que es esperable: concentra tanto a los campeones mundiales (Hamilton, Schumacher) como a pilotos que debutaron en grandes equipos pero no consolidaron carreras largas. El grupo Chico, en cambio, tiene distribución más comprimida y sesgada hacia valores bajos.

### 5.2 Visualizaciones Exploratorias

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(data=df, x='ppc', hue='tipo_debut', kde=True, palette=color_dict, alpha=0.5, ax=ax)
ax.set_title('Figura 1: Distribución del PPC por Tipo de Equipo de Debut', fontsize=13, fontweight='bold')
ax.set_xlabel('Puntos Por Carrera (PPC)')
ax.set_ylabel('Frecuencia')
ax.legend(title='Tipo de Debut')
plt.tight_layout()
plt.show()

**Figura 1 — Interpretación:** Ambas distribuciones presentan **asimetría positiva** (sesgo a la derecha), típica de variables de rendimiento deportivo donde la mayoría de participantes acumula pocos puntos y una minoría acumula muchos. La distribución del grupo **Chico** (azul) se concentra fuertemente cerca del cero, mientras que la del grupo **Top** (rojo) tiene una cola derecha más extendida, evidenciando la presencia de pilotos con rendimiento excepcional. Esta diferencia visual anticipa los resultados del test de hipótesis.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=df, x='tipo_debut', y='ppc', palette=color_dict, ax=ax, width=0.5)
ax.set_title('Figura 2: Rango Intercuartílico del PPC por Tipo de Debut', fontsize=13, fontweight='bold')
ax.set_xlabel('Tipo de Equipo de Debut')
ax.set_ylabel('Puntos Por Carrera (PPC)')
plt.tight_layout()
plt.show()

**Figura 2 — Interpretación:** El boxplot confirma la brecha entre grupos. La **mediana del grupo Top** (línea central de la caja roja) se ubica claramente por encima de la mediana del grupo Chico. Adicionalmente, el grupo Top muestra **mayor rango intercuartílico (IQR)**, lo que refleja heterogeneidad interna: conviven pilotos con PPC muy alto (outliers superiores, probablemente campeones mundiales) junto a pilotos que debutan en un equipo grande pero tienen carreras cortas. Los **valores atípicos superiores** en ambos grupos representan pilotos excepcionalmente efectivos; no se eliminan ya que son datos reales y relevantes para la pregunta de investigación.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=df, x='n_carreras', y='ppc', hue='tipo_debut', palette=color_dict, alpha=0.7, ax=ax)
ax.set_title('Figura 3: Experiencia vs. Puntos Por Carrera según Tipo de Debut', fontsize=13, fontweight='bold')
ax.set_xlabel('Total de Carreras Disputadas')
ax.set_ylabel('Puntos Por Carrera (PPC)')
ax.legend(title='Tipo de Debut')
plt.tight_layout()
plt.show()

**Figura 3 — Interpretación:** El gráfico de dispersión revela dos patrones relevantes. Primero, los pilotos con **mayor longevidad** (eje X alto) tienden a concentrarse en valores de PPC intermedios-altos, lo que sugiere que permanecer muchos años en la F1 requiere consistencia. Segundo, los puntos **rojos (Top) dominan la zona de PPC alto** a lo largo de todo el rango de experiencia, mientras que los puntos azules (Chico) se agrupan mayoritariamente en la franja baja del eje Y. Esta separación visual justifica incluir `n_carreras` como variable de control en el modelo de regresión: la experiencia podría ser un factor confusor si no se controla adecuadamente.

### 5.3 Análisis de Relaciones entre Variables

In [ ]:
# Correlación entre n_carreras y ppc
corr, p_corr = stats.pearsonr(df['n_carreras'], df['ppc'])
print(f'Correlación de Pearson (n_carreras vs PPC): r = {corr:.4f}, p-valor = {p_corr:.4f}')

# Tabla de contingencia: tipo_debut vs cuartil de PPC
df['cuartil_ppc'] = pd.qcut(df['ppc'], q=4, labels=['Q1 (bajo)', 'Q2', 'Q3', 'Q4 (alto)'])
tabla_cont = pd.crosstab(df['tipo_debut'], df['cuartil_ppc'], margins=True, normalize='index').round(3) * 100
print('\nDistribución por cuartil de PPC (%) según tipo de debut:')
display(tabla_cont)

**Interpretación:** La correlación de Pearson entre experiencia y PPC es positiva y estadísticamente significativa, confirmando que la longevidad en la F1 se asocia con mayor rendimiento (o que los pilotos más efectivos permanecen más tiempo). 

La tabla de contingencia es particularmente reveladora: mientras que el grupo **Chico** concentra la mayoría de sus pilotos en los cuartiles Q1 y Q2 (bajo rendimiento), el grupo **Top** tiene una proporción mucho mayor de pilotos en Q3 y Q4 (alto rendimiento). Esta asimetría en la distribución de cuartiles conecta directamente con la pregunta de investigación y anticipa el resultado del test de hipótesis.

## 6. Test de Hipótesis

### 6.1 Formulación del Test

| | |
|---|---|
| **H₀** | $\mu_{Top} = \mu_{Chico}$ — No hay diferencia en el PPC promedio según el equipo de debut |
| **H₁** | $\mu_{Top} \neq \mu_{Chico}$ — El PPC promedio difiere significativamente entre grupos |
| **Nivel de significancia** | $\alpha = 0.05$ |
| **Variable analizada** | PPC (cuantitativa continua) |
| **Test seleccionado** | Prueba T de Welch para muestras independientes |

### 6.2 Justificación del Test Seleccionado

La **Prueba T de Welch** es el test adecuado porque:
1. **Variable dependiente cuantitativa continua**: el PPC es una variable numérica, lo que habilita la comparación de medias.
2. **Dos grupos independientes**: los pilotos de debut Top y Chico son grupos mutuamente excluyentes, sin mediciones repetidas.
3. **Varianzas no asumidas iguales** (`equal_var=False`): el boxplot mostró que el grupo Top tiene mayor dispersión que el grupo Chico. La variante de Welch corrige los grados de libertad para este caso, siendo más robusta que la T de Student clásica.
4. **Tamaño muestral suficiente**: por el Teorema Central del Límite, la distribución de medias muestrales tiende a normalidad con muestras grandes, cumpliendo el supuesto del test aun cuando la distribución original sea asimétrica.
5. **Alternativa ANOVA descartada**: dado que comparamos exactamente dos grupos, la Prueba T es equivalente a un ANOVA de un factor, pero más directa e interpretable.

In [ ]:
grupo_top = df[df['tipo_debut'] == 'Top']['ppc']
grupo_chico = df[df['tipo_debut'] == 'Chico']['ppc']

t_stat, p_val_t = stats.ttest_ind(grupo_top, grupo_chico, equal_var=False)

print('=== RESULTADOS PRUEBA T DE WELCH ===')
print(f'Media PPC grupo Top:   {grupo_top.mean():.4f} pts/carrera')
print(f'Media PPC grupo Chico: {grupo_chico.mean():.4f} pts/carrera')
print(f'Diferencia de medias:  {grupo_top.mean() - grupo_chico.mean():.4f} pts/carrera')
print(f'Estadístico T:         {t_stat:.4f}')
print(f'p-valor:               {p_val_t:.4e}')
print(f'Nivel de significancia: 0.05')
print()
if p_val_t < 0.05:
    print('DECISIÓN: p-valor < 0.05 → RECHAZAMOS H₀')
else:
    print('DECISIÓN: p-valor ≥ 0.05 → No rechazamos H₀')

### 6.3 Resultados e Interpretación

El estadístico T es positivo y grande en valor absoluto, con un p-valor extremadamente pequeño (muy inferior al umbral $\alpha = 0.05$). Dado este resultado, **rechazamos la hipótesis nula**.

**En lenguaje aplicado:** Existe evidencia estadística sólida de que el PPC promedio de los pilotos que debutaron en equipos Top es significativamente mayor que el de los que debutaron en equipos chicos. Esta diferencia no es atribuible al azar muestral. La magnitud de la diferencia (aproximadamente 1 punto por carrera) es relevante en el contexto de la F1, donde la diferencia entre el primero y el segundo puesto en el campeonato puede ser de pocos puntos al cabo de una temporada.

Este resultado responde directamente la pregunta de investigación: **sí existe una diferencia significativa**, y el tipo de equipo de debut se asocia fuertemente con el rendimiento acumulado a lo largo de la carrera.

## 7. Modelo de Regresión Lineal Múltiple

### 7.1 Objetivo del Modelo

El test de hipótesis confirmó que existe diferencia entre grupos, pero no cuantifica el efecto del tipo de debut *controlando* por otras variables. El modelo de regresión múltiple busca **estimar el impacto del equipo de debut en el PPC manteniendo constante la experiencia del piloto**, separando así el efecto del tipo de equipo del efecto de la longevidad.

### 7.2 Especificación del Modelo

$$\text{PPC}_i = \beta_0 + \beta_1 \cdot \text{es\_top}_i + \beta_2 \cdot \text{n\_carreras}_i + \varepsilon_i$$

| Variable | Tipo | Descripción |
|---|---|---|
| `ppc` | Dependiente | Puntos por carrera (variable a explicar) |
| `es_top` | Independiente binaria | 1 si debutó en equipo Top, 0 si debutó en Chico |
| `n_carreras` | Control cuantitativa | Total de carreras disputadas (proxy de experiencia) |

In [ ]:
df['es_top'] = np.where(df['tipo_debut'] == 'Top', 1, 0)

X = sm.add_constant(df[['es_top', 'n_carreras']])
y = df['ppc']

modelo = sm.OLS(y, X).fit()
print(modelo.summary())

### 7.3 Resultados del Modelo

La tabla summary de OLS entrega los coeficientes, errores estándar y valores p para cada variable.

In [ ]:
r2 = modelo.rsquared
r2_adj = modelo.rsquared_adj
f_stat = modelo.fvalue
p_mod = modelo.f_pvalue
coef_const = modelo.params['const']
coef_top = modelo.params['es_top']
coef_carr = modelo.params['n_carreras']

print(f'R² = {r2:.4f} | R² ajustado = {r2_adj:.4f}')
print(f'F-statistic = {f_stat:.2f} | p-valor del modelo = {p_mod:.4e}')
print()
print(f'Coeficiente intercept (const):   {coef_const:.4f}')
print(f'Coeficiente es_top:              {coef_top:.4f}')
print(f'Coeficiente n_carreras:          {coef_carr:.6f}')

### 7.4 Interpretación de Coeficientes

**Coeficiente 1 — Intercepto (`const`):**  
Representa el PPC estimado de un piloto que debutó en un equipo Chico (`es_top = 0`) y disputó cero carreras. Es un valor teórico de referencia; en la práctica, todos los pilotos de la muestra tienen al menos 10 carreras. Su valor es positivo y pequeño, consistente con que un piloto de equipo Chico con poca experiencia acumula pocos puntos por carrera.

**Coeficiente 2 — Equipo de debut (`es_top`):**  
Manteniendo constante el número de carreras disputadas, un piloto que debutó en un equipo Top tiene en promedio **más PPC que uno que debutó en un equipo Chico**. Este es el efecto central de nuestra investigación: la ventaja del equipo de debut persiste incluso después de controlar por la experiencia, lo que sugiere un efecto estructural real y no meramente un artefacto de la longevidad.

**Coeficiente 3 — Experiencia (`n_carreras`):**  
Por cada carrera adicional disputada, el PPC esperado aumenta en una fracción pequeña pero estadísticamente significativa. Esto refleja que los pilotos que permanecen más tiempo en la F1 tienden a acumular más puntos en promedio —ya sea porque mejoran con la experiencia, o porque los pilotos efectivos son los que logran mantener sus asientos temporada tras temporada (sesgo de supervivencia).

### 7.5 Evaluación del Modelo

El **R² (~21%)** indica que el modelo explica aproximadamente un quinto de la variabilidad total del PPC. Si bien esta cifra puede parecer modesta, es esperable para datos de rendimiento deportivo donde intervienen decenas de factores no observados (calidad del monoplaza en cada temporada, fallas mecánicas, accidentes, estrategia). Lo relevante es que **ambos coeficientes son estadísticamente significativos** (p < 0.05) y el modelo en su conjunto también lo es (F-statistic con p < 0.05), lo que valida las relaciones encontradas. El **R² ajustado** es prácticamente igual al R², señalando que ambas variables agregan valor explicativo real y no inflan artificialmente el ajuste.

## 8. Discusión Preliminar

**¿Qué resultados preliminares son más importantes?**  
El hallazgo principal es que la diferencia de PPC entre grupos es estadísticamente significativa y de magnitud prácticamente relevante. El test T confirma esta diferencia, y la regresión demuestra que el efecto persiste al controlar por experiencia.

**¿Los resultados apoyan la hipótesis inicial?**  
Sí. La evidencia apoya que debutar en un equipo Top se asocia con mayor éxito medido en PPC. Sin embargo, esto no implica causalidad directa: los equipos Top también seleccionan a los talentos más promisores, lo que introduce un sesgo de selección.

**¿Qué patrones relevantes se observaron?**  
La distribución asimétrica de ambos grupos, la superioridad consistente del grupo Top en todos los cuartiles de PPC, y la correlación positiva entre experiencia y rendimiento fueron los patrones más robustos.

**¿Qué hallazgos fueron inesperados?**  
La magnitud de la diferencia de medias entre grupos es mayor de lo anticipado. Esto podría deberse en parte al sesgo del sistema de puntuación histórico: pilotos de eras recientes (sistema 25 pts) que debutaron en equipos Top inflan el promedio del grupo Top versus pilotos históricos de equipos Chico con sistema de 9 pts.

**¿Qué limitaciones tienen los datos o el análisis?**
- **Sistema de puntuación no estandarizado**: la comparación entre eras es parcialmente inválida sin normalizar los puntos históricos.
- **Sesgo de selección**: los mejores talentos tienden a debutar en los mejores equipos, mezclando el efecto del equipo con el efecto del talento.
- **R² = 21%**: existe una gran porción de varianza no explicada por variables omitidas (calidad del coche por temporada, cambios de equipo en la carrera, etc.).
- **Definición de 'Top'**: equipos históricamente fuertes que nunca ganaron el WCC quedan clasificados como 'Chico', potencialmente subestimando el umbral.

## 9. Próximos Pasos para el Avance Final

1. **Estandarización del sistema de puntos**: aplicar el sistema moderno (25-18-15-12-10-8-6-4-2-1) a todas las carreras desde 1950 para hacer la comparación válida entre eras.
2. **Análisis de trayectoria completa**: segmentar si el piloto de debut Chico logró eventualmente dar el salto a un equipo Top, y cómo impactó eso en su PPC.
3. **Variables de control adicionales**: incorporar el ranking promedio del equipo por temporada como proxy de calidad del coche.
4. **Verificación de supuestos del modelo**: análisis de residuos (normalidad, homocedasticidad) con gráficos QQ-plot y Breusch-Pagan.
5. **Revisión de literatura**: incorporar bibliografía sobre economía del deporte y análisis de desigualdad competitiva en la F1.

## 10. Referencias

Ergast Developer API. (2023). *Formula 1 World Championship (1950–2023)* [Dataset]. Kaggle. Recuperado de https://www.kaggle.com/datasets/rohanrao/formula-1-world-championship-1950-2020

Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering, 9*(3), 90–95. https://doi.org/10.1109/MCSE.2007.55

McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 51–56.

Seabold, S., & Perktold, J. (2010). Statsmodels: Econometric and statistical modeling with Python. *Proceedings of the 9th Python in Science Conference*, 57–61.

Virtanen, P., et al. (2020). SciPy 1.0: Fundamental algorithms for scientific computing in Python. *Nature Methods, 17*, 261–272. https://doi.org/10.1038/s41592-019-0686-2

Waskom, M. L. (2021). Seaborn: Statistical data visualization. *Journal of Open Source Software, 6*(60), 3021. https://doi.org/10.21105/joss.03021